# 📧 Email Triage RL Environment — Complete Demo & Test

**OpenEnv Hackathon 2026 · Team Ctrl-Alt-Defeat**  
Haraprasad Hota · Subhendu Samal · Ashutosh Panigrahi

[![Open in Spaces](https://huggingface.co/datasets/huggingface/badges/resolve/main/open-in-hf-spaces-sm-dark.svg)](https://huggingface.co/spaces/Hk4crprasad/email-triage-env)

This notebook demonstrates the **complete pipeline** end-to-end:

| Section | What it covers |
|---------|----------------|
| 1 | Install & clone the environment |
| 2 | Run the 26-check validation suite |
| 3 | Live episode walkthrough (step by step) |
| 4 | Run baseline inference with any LLM (API mode) |
| 5 | Run inference with our trained LoRA adapter |
| 6 | Reward breakdown visualization |
| 7 | Baseline vs. trained comparison plot |

> **No GPU required** for Sections 1–4.  GPU recommended for Section 5.

## 🔧 Section 1: Install & Setup

In [ ]:
%%capture
# Install runtime dependencies
!pip install fastapi uvicorn pydantic openai requests motor gradio matplotlib
!pip install openenv>=0.1.13

# Clone the environment repo from HF Space
!git clone https://huggingface.co/spaces/Hk4crprasad/email-triage-env /content/email-triage-env 2>/dev/null || echo 'Already cloned'

import sys
sys.path.insert(0, '/content/email-triage-env')
print('✅ Setup complete')

In [ ]:
import sys, os
sys.path.insert(0, '/content/email-triage-env')

# Verify all core modules load cleanly
from server.environment import EmailTriageEnvironment
from server.email_generator import generate_emails
from server.tasks import TASKS, list_task_ids
from server.reward import REWARD_RUBRIC
from server.graders import grade_episode, get_rubric_definitions
from models import EmailAction, EmailObservation

print('✅ All modules imported successfully')
print(f'   Tasks: {list_task_ids()}')
print(f'   Reward components: {list(REWARD_RUBRIC.keys())}')

## ✅ Section 2: Validation Suite (26 checks)

In [ ]:
import subprocess
result = subprocess.run(
    ['python', 'scripts/validate_env.py'],
    capture_output=True, text=True,
    cwd='/content/email-triage-env'
)
print(result.stdout)
if result.returncode == 0:
    print('🎉 All checks passed!')
else:
    print('❌ Some checks failed:')
    print(result.stderr)

## 🎮 Section 3: Live Episode Walkthrough (step by step)

In [ ]:
import json

# ── Choose task: 'easy' | 'medium' | 'hard' ──
TASK_ID = 'easy'
SEED = 42

env = EmailTriageEnvironment()
obs = env.reset(task_id=TASK_ID, seed=SEED)
obs_dict = obs.model_dump()

task_info = TASKS[TASK_ID]
print(f'Task: {task_info.name}')
print(f'Emails: {obs_dict["inbox_stats"]["total"]}')
print(f'Max steps: {task_info.max_steps}')
print(f'Scoring weights: {task_info.scoring_weights}')
print(f'\n--- INBOX ---')
for e in obs_dict['emails']:
    print(f'  [{e["email_id"]}] From: {e["sender"]} | Subject: {e["subject"][:60]}')

In [ ]:
# ── Show the first email in detail ──
email = obs_dict['emails'][0]
print(f'=== EMAIL: {email["email_id"]} ===')
print(f'From    : {email["sender_name"]} <{email["sender"]}>')
print(f'Subject : {email["subject"]}')
print(f'Time    : {email["timestamp"]}')
print(f'Reply   : {email["is_reply"]} | Attachment: {email["has_attachment"]}')
print(f'\nBody:\n{email["body"]}')

In [ ]:
# ── Submit a CORRECT action and see the reward ──
emails_gt, ground_truths = generate_emails(TASK_ID, SEED)
truth_map = {gt.email_id: gt for gt in ground_truths}

email_id = email['email_id']
gt = truth_map[email_id]

correct_action = {
    'email_id': email_id,
    'category': gt.category,
    'priority': gt.priority,
    'department': gt.department,
    'response_draft': ' '.join(gt.expected_keywords) if gt.requires_response else None,
    'escalate': gt.department == 'management' or gt.priority == 1,
}

print(f'Ground truth: category={gt.category}, priority={gt.priority}, dept={gt.department}')
print(f'Action: {json.dumps(correct_action, indent=2)}')

obs = env.step(correct_action)
obs_dict = obs.model_dump()
print(f'\n✅ Step reward : {obs_dict["step_reward"]:+.4f}')
print(f'   Feedback    : {obs_dict["action_feedback"]}')
print(f'   Cumulative  : {obs_dict["cumulative_reward"]:+.4f}')

In [ ]:
# ── Submit a WRONG action — show penalty ──
if obs_dict['emails']:  # next email
    next_email = obs_dict['emails'][0]
    wrong_action = {
        'email_id': next_email['email_id'],
        'category': 'urgent',    # deliberately wrong
        'priority': 1,           # deliberately wrong
        'department': 'engineering',
        'escalate': True,
    }
    obs2 = env.step(wrong_action)
    o2 = obs2.model_dump()
    gt2 = truth_map[next_email['email_id']]
    print(f'Wrong action on {next_email["email_id"]}')
    print(f'Ground truth: category={gt2.category}, priority={gt2.priority}')
    print(f'❌ Step reward : {o2["step_reward"]:+.4f}')
    print(f'   Feedback    : {o2["action_feedback"]}')

In [ ]:
# ── Run a PERFECT episode (all ground-truth actions) ──
env2 = EmailTriageEnvironment()
obs = env2.reset(task_id=TASK_ID, seed=SEED)
emails_gt, ground_truths = generate_emails(TASK_ID, SEED)
truth_map = {gt.email_id: gt for gt in ground_truths}

step_rewards = []
for email in emails_gt:
    gt = truth_map[email.email_id]
    action = {
        'email_id': email.email_id,
        'category': gt.category,
        'priority': gt.priority,
        'department': gt.department,
        'response_draft': ' '.join(gt.expected_keywords) if gt.requires_response and gt.expected_keywords else None,
        'escalate': gt.department == 'management' or gt.priority == 1,
    }
    obs = env2.step(action)
    o = obs.model_dump()
    step_rewards.append(o['step_reward'])
    print(f'  [{email.email_id}] reward={o["step_reward"]:+.4f} | {o["action_feedback"][:60]}')

grading = o['metadata']['grading']
print(f'\n🏁 Episode complete!')
print(f'   Final score     : {grading["final_score"]:.4f}')
print(f'   Emails processed: {grading["emails_processed"]}/{grading["emails_total"]}')
print(f'   Dimensions      : {json.dumps(grading["dimension_scores"], indent=4)}')

## 🤖 Section 4: Baseline Inference — Any LLM via API

Uses the HF Inference Router (free) with `openai/gpt-oss-120b`.  
Set your HF token below.

In [ ]:
# 🔑 Set your HuggingFace token here
import os
os.environ['HF_TOKEN'] = 'hf_...'                         # ← paste your token
os.environ['MODEL_NAME'] = 'openai/gpt-oss-120b'          # free on HF router
os.environ['API_BASE_URL'] = 'https://router.huggingface.co/v1'
os.environ['USE_LOCAL_MODEL'] = '0'
print('Token set:', os.environ['HF_TOKEN'][:10] + '...')

In [ ]:
# Run baseline inference — all 3 tasks
import subprocess
result = subprocess.run(
    ['python', 'inference.py'],
    capture_output=True, text=True,
    cwd='/content/email-triage-env',
    env={**os.environ}
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:500])

## 🧠 Section 5: Inference with Trained LoRA Adapter

Loads our GRPO-trained adapter (`Hk4crprasad/email-triage-grpo`) on top of `Qwen/Qwen2.5-3B-Instruct`.

> **Requires GPU** (T4 or better). Enable via Runtime → Change runtime type → T4 GPU.

In [ ]:
%%capture
!pip install transformers>=4.40.0 peft>=0.10.0 accelerate>=0.28.0 bitsandbytes>=0.43.0
print('✅ Adapter inference deps installed')

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ No GPU — adapter inference will be slow (CPU). Switch runtime for T4 GPU.')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL_ID    = 'Qwen/Qwen2.5-3B-Instruct'
ADAPTER_MODEL_ID = 'Hk4crprasad/email-triage-grpo'
HF_TOKEN         = os.environ.get('HF_TOKEN')

print(f'Loading base model  : {BASE_MODEL_ID}')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
    token=HF_TOKEN,
    trust_remote_code=True,
)

print(f'Loading LoRA adapter: {ADAPTER_MODEL_ID}')
model = PeftModel.from_pretrained(base_model, ADAPTER_MODEL_ID, token=HF_TOKEN)
model.eval()
print(f'✅ Adapter loaded! Device: {next(model.parameters()).device}')

In [ ]:
# Run the trained adapter against all 3 tasks
os.environ['USE_LOCAL_MODEL'] = '1'
os.environ['ADAPTER_MODEL_ID'] = ADAPTER_MODEL_ID
os.environ['BASE_MODEL_ID'] = BASE_MODEL_ID

result = subprocess.run(
    ['python', 'inference.py'],
    capture_output=True, text=True,
    cwd='/content/email-triage-env',
    env={**os.environ}
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:500])

## 📊 Section 6: Reward Breakdown Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
matplotlib.rcParams['figure.dpi'] = 120

# ── Run a perfect vs random episode and capture per-component rewards ──
from server.reward import (
    reward_classification, reward_priority, reward_routing,
    reward_response_quality, reward_escalation, reward_format_compliance
)

TASK = 'hard'
emails_gt, ground_truths = generate_emails(TASK, 42)
truth_map = {gt.email_id: gt for gt in ground_truths}
valid_ids = {e.email_id for e in emails_gt}

perfect_scores = {'classification': [], 'priority': [], 'routing': [], 'response': [], 'escalation': [], 'format': []}
random_scores  = {'classification': [], 'priority': [], 'routing': [], 'response': [], 'escalation': [], 'format': []}

import random
rng = random.Random(0)
CATS = ['spam', 'billing', 'technical', 'general', 'urgent']
DEPTS = ['engineering', 'billing', 'support', 'management']

for email in emails_gt:
    gt = truth_map[email.email_id]
    perfect = {'email_id': email.email_id, 'category': gt.category,
               'priority': gt.priority, 'department': gt.department,
               'response_draft': ' '.join(gt.expected_keywords) if gt.requires_response else None,
               'escalate': gt.department == 'management' or gt.priority == 1}
    rand_action = {'email_id': email.email_id, 'category': rng.choice(CATS),
                   'priority': rng.randint(1, 5), 'department': rng.choice(DEPTS),
                   'response_draft': None, 'escalate': rng.random() > 0.7}

    perfect_scores['classification'].append(reward_classification(perfect, gt))
    perfect_scores['priority'].append(reward_priority(perfect, gt))
    perfect_scores['routing'].append(reward_routing(perfect, gt))
    perfect_scores['response'].append(reward_response_quality(perfect, gt))
    perfect_scores['escalation'].append(reward_escalation(perfect, gt))
    perfect_scores['format'].append(reward_format_compliance(perfect, valid_ids))

    random_scores['classification'].append(reward_classification(rand_action, gt))
    random_scores['priority'].append(reward_priority(rand_action, gt))
    random_scores['routing'].append(reward_routing(rand_action, gt))
    random_scores['response'].append(reward_response_quality(rand_action, gt))
    random_scores['escalation'].append(reward_escalation(rand_action, gt))
    random_scores['format'].append(reward_format_compliance(rand_action, valid_ids))

components = list(perfect_scores.keys())
p_means = [np.mean(perfect_scores[c]) for c in components]
r_means = [np.mean(random_scores[c]) for c in components]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(components))
w = 0.35
ax.bar(x - w/2, p_means, w, label='Perfect actions', color='#2ecc71', alpha=0.85)
ax.bar(x + w/2, r_means, w, label='Random actions',  color='#e74c3c', alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(components, fontsize=11)
ax.set_ylabel('Mean reward per email', fontsize=11)
ax.set_title('Reward Component Separation — HARD task (7 independent verifiers)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('/content/reward_spread.png', bbox_inches='tight')
plt.show()
print('\nMean rewards — perfect vs random:')
for c, p, r in zip(components, p_means, r_means):
    print(f'  {c:<16} perfect={p:+.3f}   random={r:+.3f}')

## 📈 Section 7: Baseline vs. Trained — Score Comparison

In [ ]:
# Published results from our GRPO training run
# (Run Section 4 and 5 to get live values; these are our published numbers)
results = {
    'Task':     ['easy',  'medium', 'hard'],
    'Baseline': [0.60,    0.38,     0.29],
    'GRPO':     [0.92,    0.64,     0.51],
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Score comparison bar chart ──
x = np.arange(3)
w = 0.35
axes[0].bar(x - w/2, results['Baseline'], w, label='Baseline (0-shot)', color='#95a5a6', alpha=0.9)
axes[0].bar(x + w/2, results['GRPO'],     w, label='After GRPO (ours)', color='#2980b9', alpha=0.9)
for i, (b, g) in enumerate(zip(results['Baseline'], results['GRPO'])):
    axes[0].annotate(f'+{g-b:.2f}', xy=(i + w/2, g + 0.01), ha='center', fontsize=10, color='#27ae60', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(results['Task'], fontsize=12)
axes[0].set_ylabel('Final Score (0→1)', fontsize=11)
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Baseline vs. Trained — Score by Task', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# ── Improvement bar chart ──
improvements = [g - b for g, b in zip(results['GRPO'], results['Baseline'])]
colors = ['#27ae60' if v > 0 else '#e74c3c' for v in improvements]
bars = axes[1].bar(results['Task'], improvements, color=colors, alpha=0.85)
for bar, val in zip(bars, improvements):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'+{val:.2f}', ha='center', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Score Improvement (Δ)', fontsize=11)
axes[1].set_title('GRPO Improvement over 0-shot Baseline', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim(0, 0.45)

plt.tight_layout()
plt.savefig('/content/score_comparison.png', bbox_inches='tight')
plt.show()

avg_improvement = sum(improvements) / len(improvements)
print(f'Average improvement: +{avg_improvement:.3f}')

In [ ]:
# ── Per-dimension improvement on medium task ──
dims = {
    'classification': {'baseline': 0.48, 'trained': 0.72},
    'priority':       {'baseline': 0.31, 'trained': 0.55},
    'routing':        {'baseline': 0.29, 'trained': 0.50},
}

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(dims))
w = 0.35
b_vals = [v['baseline'] for v in dims.values()]
t_vals = [v['trained']  for v in dims.values()]

ax.bar(x - w/2, b_vals, w, label='Baseline (0-shot)', color='#95a5a6', alpha=0.9)
ax.bar(x + w/2, t_vals, w, label='After GRPO (ours)', color='#2980b9', alpha=0.9)
for i, (b, t) in enumerate(zip(b_vals, t_vals)):
    delta = t - b
    ax.annotate(f'+{delta:.2f}', xy=(i + w/2, t + 0.01), ha='center', fontsize=10,
                color='#27ae60', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(list(dims.keys()), fontsize=12)
ax.set_ylabel('Accuracy (0→1)', fontsize=11)
ax.set_title('Per-Dimension Improvement — Medium Task', fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(0, 0.85)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('/content/dimension_breakdown.png', bbox_inches='tight')
plt.show()

## 🔗 Links & Resources

| Resource | URL |
|----------|-----|
| HF Space (live API) | https://huggingface.co/spaces/Hk4crprasad/email-triage-env |
| Trained adapter | https://huggingface.co/Hk4crprasad/email-triage-grpo |
| GitHub | https://github.com/hk4crprasad/my-env |
| GRPO Training notebook | `notebooks/train_grpo.ipynb` |

### Quick command reference
```bash
# Validate environment locally (26 checks)
python scripts/validate_env.py

# Run API inference
HF_TOKEN=hf_... python inference.py

# Run trained adapter inference
HF_TOKEN=hf_... USE_LOCAL_MODEL=1 python inference.py

# Start the server
uvicorn server.app:app --host 0.0.0.0 --port 7860

# Launch interactive demo
python demo.py
```